# 02 — answer the centre probe with one model

Rung 48. One model per run (papermill `-p MODEL …`), so two can share the box on the two GPUs.

**The probe:** 2,445 positive frames over the **15 held-out** CholecT50 videos, one question each
— the challenge's own most frequent `fo_class` template, verbatim. Metric is **containment**
(`gold ⊆ predicted`), because CholecT50 never annotates gauze, needles or drains and a missing
label is not an absent object.

🔴 **Recall alone can be gamed**: answering every class to every frame scores 100 %. `mean_set_size`
comes back beside it and must be read first.

**What each model's number means:**

| model | platform score | what `hold` measures for it |
|---|---|---|
| rung 06 ep2 | **0.4767** | unseen centre |
| A2 ep3 | **0.5288** | unseen centre — adapter recovered from the RunPod network volume 2026-08-19 |
| rung 42 ep4 | **0.5809** | unseen centre |
| rung 47 ep4 | — | unseen centre |
| 27B conn4e5 | — | unseen centre · **different pipeline, not this notebook** |

The three with a platform score are the calibration: **if the probe ranks them the way the
leaderboard did, it has ordinal validity on the axis that scores.** If it does not, it is not a
ruler and nothing should be hung on it.

In [ ]:
# --- bootstrap -------------------------------------------------------------------
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")
print("python:", sys.executable)

In [ ]:
# --- parameters (RAW LITERALS ONLY) -----------------------------------------------
MODEL   = "r42"          # r06 | r42 | r47
LIMIT   = 20             # None for the full 2,445
GPU     = "0"
STORAGE = "/mnt/storage/uaq_user"

In [ ]:
# --- derived ----------------------------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
os.environ["HF_HOME"] = f"{STORAGE}/hf_cache"
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

REPO = f"{STORAGE}/repo_leo"
EXP  = Path(REPO) / "experiments" / "48-centre-probe"
sys.path.insert(0, str(EXP / "_tools"))
# 🔴 `focus` is VENDORED in the repo, not installed in this env. Rung 40 lost a cycle
# "fixing" that by switching envs. Reuse rung 45's own path setup — imported, never copied.
sys.path.insert(0, f"{REPO}/experiments/45-gen36-data-and-reg/_tools")
from eval_arm45 import ensure_paths
ensure_paths(REPO)
import cholect50 as T
import probe_runner as R

BASE = next(Path(f"{STORAGE}/hf_cache/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots").glob("*"))
WORK = Path(f"{STORAGE}/rung48")

# adapter (or checkpoint) per model, and what its platform score was
MODELS = {
    "r06": {"adapter": WORK / "adapters/rung06_ep2",                     "platform": 0.4767},
    "a2":  {"adapter": WORK / "adapters/a2_ep3",                          "platform": 0.5288},
    "r42": {"adapter": Path(f"{STORAGE}/rung19/r42_ep4_adapter"),        "platform": 0.5809},
    "r47": {"adapter": next(Path(f"{STORAGE}/rung47/runs/47_a2_ep5_v1/ckpt").glob("v0-*"))
                       / "checkpoint-3604",                              "platform": None},
    # rung 14 ep2 (appearance augmentation). Its control is r06, NOT a2: `args.json`
    # records lr 2e-5, r8/a32, freeze_aligner=true, target_modules=[all-linear], seed 42
    # -- rung 06's recipe exactly, differing only in `--dataset` (train_aug.jsonl).
    # Read `bag_f1` against r06's 0.8453. It has no platform score and never will.
    "r14": {"adapter": WORK / "adapters/r14_ep2",                        "platform": None},
    # Rung 42's OTHER epochs. `r42` above is ep4 (checkpoint-4848) — the checkpoint that
    # shipped as submission 03 — and it was selected by the held-out sweep in
    # `42-merged-corpus/01_eval_heldout.ipynb`, i.e. by the LOCAL instrument, which
    # inverts on `object_recognition_OOD` and overstates it by +0.367
    # ([[local-eval-vs-judge-calibration]]). OOD is half of `bucket_mean`, so epoch
    # selection has never been done on a cell that can read centre. These three make the
    # curve readable on `bag_f1`. Same run, same recipe, same seed — the epoch is the
    # only variable, and no training is involved.
    "r42_ep2": {"adapter": WORK / "adapters/r42_ep2",                    "platform": None},
    "r42_ep3": {"adapter": WORK / "adapters/r42_ep3",                    "platform": None},
    "r42_ep5": {"adapter": WORK / "adapters/r42_ep5",                    "platform": None},
    # Rung 50 arm B, epoch 4 — the best `bucket_mean` on the full 6,252 of any arm that
    # can be scored there (0.6594), winning or tying 3 of its 4 buckets. It was CLOSED as
    # a faithful negative, but on a MECHANISM test: its gain sat at gold set size 1, so it
    # failed as evidence for the enumeration hypothesis. Its own note records that the
    # direction is consistent 4/4 and "should not be recorded as nothing"
    # ([[enumeration-is-not-fixed-by-output-format]]). It was rejected as an explanation,
    # not as a model.
    # 🔴 WHY IT NEEDS THIS PROBE AT ALL: rung 42 trained on 30 of the 38 public test
    # videos, so it CANNOT be scored on the full 6,252 and 0.6594 vs 0.6744 is not a
    # comparison. CholecT50 is external to both, so this is the only common ground.
    # It saw no CholecT50, so `hold` measures centre for it (unlike 19b).
    "r50b_ep4": {"adapter": Path(f"{STORAGE}/rung50/runs/50b_contweight_v1/ckpt")
                            / "v1-20260823-122952" / "checkpoint-3604",  "platform": None},
}
M = MODELS[MODEL]
MERGED = WORK / "merged" / MODEL
OUT    = WORK / "runs" / MODEL
OUT.mkdir(parents=True, exist_ok=True)

print("model   :", MODEL, "· platform", M["platform"])
print("adapter :", M["adapter"], "->", M["adapter"].exists())
print("base    :", BASE)
print("out     :", OUT)

In [ ]:
# --- GATE: the items and their frames, before anything expensive. RAISES. ----------
items = pd.read_json(WORK / "corpus/probe_items_v2.jsonl", lines=True)  # v2: + negatives
sp = T.load_manifest(Path(REPO) / "experiments/splits/cholect50_split_v1.csv")
hold = set(sp[sp.split == "hold"].video_id)

if set(items.video) != hold:
    raise AssertionError(f"items cover {len(set(items.video))} videos, hold has {len(hold)}")
if len(items) != 4890:
    raise AssertionError(f"{len(items)} items, expected 4,890 (2,445 pos + 2,445 neg)")
if set(items.kind) != {"pos", "neg"}:
    raise AssertionError(f"kinds are {set(items.kind)}")
missing = [f"cholect50__{r.video}__{r.frame:06d}.jpg" for r in items.itertuples()
           if not (WORK / "frames" / f"cholect50__{r.video}__{r.frame:06d}.jpg").exists()]
if missing:
    raise AssertionError(f"{len(missing)} frames not cached, e.g. {missing[:3]}")

print(f"OK items: {len(items)} over {items.video.nunique()} videos · every frame cached")
print(items.kind.value_counts().to_string())
for c in ("clip_state", "bag_state"):
    print(" ", c, items[c].value_counts().to_dict())

In [ ]:
# --- merge -> answer -> reclaim ----------------------------------------------------
t0 = time.perf_counter()
R.merge_adapter(BASE, M["adapter"], MERGED)
pred = R.answer_items(MERGED, items, WORK / "frames", limit=LIMIT)
tag = "smoke" if LIMIT else "full"
pred.to_csv(OUT / f"predictions_v2_{tag}.csv", index=False)
print(f"{len(pred)} answered in {(time.perf_counter()-t0)/60:.1f} min -> {OUT}")
print(pred[["qID", "prediction"]].head(8).to_string(index=False))

In [ ]:
# --- score: containment, and the guard that stops it being gamed --------------------
# 🔴 per class, over the frames where the timeline settles that class's state.
# `score_containment` (v1) stays in the library but is NOT used: with no negatives it
# scored a Clip-attractor model a perfect recall — 0.9936-1.0000 on all four models.
scored = T.score_per_class(pred, items.head(len(pred)))
rows = [{"cell": k, **v} for k, v in scored.items() if k != "by_video_f1"]
print(pd.DataFrame(rows).round(4).to_string(index=False))

if not LIMIT:
    (OUT / "RESULTS_probe_v2.json").write_text(json.dumps(
        {"model": MODEL, "platform": M["platform"], **scored}, indent=1))
    print("\nwritten:", OUT / "RESULTS_probe_v2.json")

mac = scored["MACRO"]
print(f"\n🎯 MACRO F1 = {mac['f1']:.4f}   (answers list {mac['mean_set_size']:.2f} classes on average)")
for c in ("Clip", "Specimen bag"):
    v = scored.get(c)
    if v:
        print(f"   {c:13} P {v['precision']:.4f}  R {v['recall']:.4f}  F1 {v['f1']:.4f}"
              f"   [tp {v['tp']} fp {v['fp']} fn {v['fn']} tn {v['tn']}]")